# Tokenization experiment — standalone Colab notebook

**Run the next missing batch:** shard **2 of 18** (zero-based), containing **9 paired jobs / 18 checkpoints**.
This is one batch of the frozen **162-pair coarse grid**, not the whole experiment.
Shards 0 and 1 are already preserved separately. No paper claims are changed by this notebook.

1. Open this `.ipynb` in [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**.
2. Select **Runtime → Change runtime type → T4 GPU**. A compatible NVIDIA driver (branch 580+) is checked before setup.
3. Leave the default batch index at `2`, then choose **Runtime → Run all**.
4. If Drive backup is enabled, approve the Drive mount prompt. At completion, keep the downloaded results ZIP and summary.

All five inputs are embedded; no companion uploads or repository checkout are needed.
Setup installs exact locked dependencies, regenerates the specified datasets, and runs the bundled checks.
Setup can take several minutes with limited output. Training duration and GPU availability are not guaranteed.

**Persistence:** Drive backup is recommended. Mounting Drive grants this notebook access to your Drive;
it writes only a newly created folder under `MyDrive/tokenization-results`.
Set `SAVE_TO_DRIVE = False` to decline, but then VM loss can erase all results before download.
Partial backups contain only completed pairs; they are **unvalidated, not automatic-resume checkpoints**.
Do not reset the Python kernel or rerun the launch cell during training. The collector needs the original live process handle.

In [ ]:
SHARD_INDEX = 2
NUM_SHARDS = 18
SAVE_TO_DRIVE = True
if type(SHARD_INDEX) is not int or not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError('SHARD_INDEX must be an integer from 0 to 17')
if NUM_SHARDS != 18 or type(SAVE_TO_DRIVE) is not bool:
    raise ValueError('Keep NUM_SHARDS = 18 and use a boolean for SAVE_TO_DRIVE')
if SHARD_INDEX < 2:
    print('Warning: shards 0 and 1 already have preserved results; this would duplicate work.')


In [ ]:
import base64
import hashlib
import json
import shutil
import tempfile
import zipfile
from pathlib import Path

def file_sha256(path: Path) -> str:
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

def write_unchanged(path: Path, payload: bytes) -> None:
    if path.is_symlink():
        raise ValueError(f'Refusing symlink: {path}')
    if path.exists():
        if not path.is_file() or path.read_bytes() != payload:
            raise FileExistsError(f'Preserving different existing file: {path}')
        return
    with path.open('xb') as handle:
        handle.write(payload)

def install_assets(assets: dict, destination: Path) -> None:
    decoded = {}
    for name, asset in assets.items():
        if Path(name).name != name or name in {'', '.', '..'} or '\\' in name:
            raise ValueError(f'Unsafe asset name: {name}')
        payload = base64.b64decode(asset['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != asset['sha256']:
            raise ValueError(f'Embedded asset hash mismatch: {name}')
        decoded[name] = payload
    destination.mkdir(parents=True, exist_ok=True)
    for name, payload in decoded.items():
        write_unchanged(destination / name, payload)

def validate_hardware(hardware: str) -> None:
    rows = hardware.strip().splitlines()
    if len(rows) != 1 or len(rows[0].split(',')) != 2:
        raise ValueError('Expected one T4 GPU; choose Runtime > Change runtime type > T4 GPU')
    name, driver = (part.strip() for part in rows[0].split(','))
    if 'T4' not in name:
        raise ValueError('Frozen experiments require a T4 GPU; do not substitute another GPU')
    if not driver.split('.')[0].isdigit() or int(driver.split('.')[0]) < 580:
        raise ValueError('This frozen CUDA 13 environment requires NVIDIA driver branch 580 or newer without separately provisioned forward compatibility. Choose a newer Colab runtime; do not change the frozen package versions.')

def load_adapter(path: Path, expected_sha256: str, variables: dict | None=None) -> dict:
    payload = path.read_bytes()
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError(f'Adapter hash mismatch: {path.name}')
    namespace = dict(variables or {})
    namespace.update({'__name__': 'tokenization_notebook_adapter', '__file__': str(path)})
    exec(compile(payload, str(path), 'exec'), namespace)
    return namespace

def safe_result_file(project: Path, relative: str) -> Path:
    candidate = Path(relative)
    path = project / candidate
    if candidate.is_absolute() or '..' in candidate.parts or '\\' in relative or (str(candidate) != relative) or path.is_symlink() or (not path.is_file()) or (not path.resolve().is_relative_to(project.resolve())):
        raise ValueError(f'Missing or unsafe result file: {relative}')
    return path

def copy_verified(source: Path, destination: Path, expected_sha256: str) -> None:
    if source.is_symlink() or file_sha256(source) != expected_sha256:
        raise ValueError(f'Source hash mismatch or symlink: {source}')
    if destination.is_symlink():
        raise ValueError(f'Refusing symlink: {destination}')
    if destination.exists():
        if not destination.is_file() or file_sha256(destination) != expected_sha256:
            raise FileExistsError(f'Preserving different existing file: {destination}')
        return
    with source.open('rb') as incoming, destination.open('xb') as outgoing:
        shutil.copyfileobj(incoming, outgoing)
    if file_sha256(destination) != expected_sha256:
        raise ValueError(f'Backup verification failed; preserve file for inspection: {destination}')

def backup_completed(project: Path, state: dict, destination: Path, *, ledger_bytes: bytes | None=None) -> Path | None:
    manifest_path = safe_result_file(project, 'artifacts/registry/theorem-faithful-coarse.json')
    if file_sha256(manifest_path) != state['manifest_sha256']:
        raise ValueError('Manifest hash mismatch during partial backup')
    manifest = json.loads(manifest_path.read_bytes())
    selected = manifest['jobs'][state['shard']::state['num_shards']]
    if ledger_bytes is None:
        ledger_bytes = safe_result_file(project, state['progress']).read_bytes()
    ledger = json.loads(ledger_bytes)
    ids = [job['job_id'] for job in selected]
    done = ledger.get('completed_jobs')
    if ledger.get('manifest_sha256') != state['manifest_sha256'] or ledger.get('shard_index') != state['shard'] or ledger.get('num_shards') != state['num_shards'] or (ledger.get('selected_jobs') != ids) or (not isinstance(done, list)) or (len(done) > len(selected)) or ([entry.get('job_id') for entry in done] != ids[:len(done)]):
        raise ValueError('Partial backup ledger does not match the frozen shard')
    if not done:
        return None
    members = []
    for job in selected[:len(done)]:
        output = Path(job['output'])
        for relative in (output, output.with_suffix('.left.pt'), output.with_suffix('.right.pt')):
            members.append((safe_result_file(project, str(relative)), str(relative)))
    target = destination / f'partial-{len(done):02d}-of-{len(selected):02d}-UNVALIDATED.zip'
    if target.exists() or target.is_symlink():
        raise FileExistsError(f'Preserving existing partial backup: {target}')
    metadata = {'status': 'partial_unvalidated', 'completed_pairs': len(done), 'automatic_resume_supported': False, 'launch_state': state}
    with tempfile.TemporaryDirectory(prefix='tokenization-backup-') as temporary:
        local = Path(temporary) / target.name
        with zipfile.ZipFile(local, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
            for source, relative in members:
                archive.write(source, arcname=relative)
            archive.writestr(state['progress'], ledger_bytes)
            archive.writestr('partial-backup.json', json.dumps(metadata, indent=2))
        copy_verified(local, target, file_sha256(local))
    return target

def save_final(summary: dict, destination: Path) -> None:
    archive = Path(summary['archive'])
    copy_verified(archive, destination / archive.name, summary['archive_sha256'])
    write_unchanged(destination / 'summary.json', (json.dumps(summary, indent=2) + '\n').encode())


In [ ]:
EMBEDDED_ASSETS = {'tokenization-pilot-bundle.zip': {'sha256': '85c44af07f6e75537b96b9b1befae4b4b584e7ba8d9a2732e290d88ef8bc7efd', 'base64': '...'}, 'tokenization-coarse-manifest.json': {'sha256': 'cffeeacb1bbc23def9f035cc4f6a3a4578112b259a0d0703d86fa2b93f333429', 'base64': '...'}, 'tokenization-coarse-learning-rates.json': {'sha256': '05dd63a4b52015e6b89e267c22355d848a4cde8741277b81915a15393b25ca7d', 'base64': '...'}, 'colab-cli-coarse.py': {'sha256': '43106d96673d2b76b3a66d041c8b2618a753bce38372594862b1196067747b59', 'base64': 'IiIiTGF1bmNoIG9uZSBjYWxsZXItc2VsZWN0ZWQgc2hhcmQgb2YgYSBmcm96ZW4gVHJhY2sgQSBjb2Fyc2UgcmVnaXN0cnkgb24gVDQuCgpFeGVjdXRlIGluIENvbGFiIGFmdGVyIHVwbG9hZGluZyB0aGUgb3JpZ2luYWwgcmVsZWFzZSBaSVAgYW5kIHNlcGFyYXRlbHkgZnJvemVuCmNvYXJzZS9MUiBKU09Ocy4gVGhpcyBhZGFwdGVyIG5laXRoZXIgYWxsb2NhdGVzIGEgcnVudGltZSBub3IgY3JlYXRlcyBhIHJlZ2lzdHJ5LgoiIiIKCmltcG9ydCBoYXNobGliCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzdGF0CmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCgpDT05URU5UX1JPT1QgPSBQYXRoKCIvY29udGVudCIpCkJVTkRMRV9OQU1FID0gInRva2VuaXphdGlvbi1waWxvdC1idW5kbGUuemlwIgpNQU5JRkVTVF9OQU1FID0gInRva2VuaXphdGlvbi1jb2Fyc2UtbWFuaWZlc3QuanNvbiIKUkFURVNfTkFNRSA9ICJ0b2tlbml6YXRpb24tY29hcnNlLWxlYXJuaW5nLXJhdGVzLmpzb24iCkJVTkRMRV9TSEEyNTYgPSAiODVjNDRhZjA3ZjZlNzU1MzdiOTZiOWIxYmVmYWU0YjRiNTg0ZTdiYThkOWEyNzMyZTI5MGQ4OGVmOGJjN2VmZCIKTUFOSUZFU1RfUEFUSCA9ICJhcnRpZmFjdHMvcmVnaXN0cnkvdGhlb3JlbS1mYWl0aGZ1bC1jb2Fyc2UuanNvbiIKUkFURVNfUEFUSCA9ICJhcnRpZmFjdHMvcmVnaXN0cnkvdGhlb3JlbS1mYWl0aGZ1bC1sZWFybmluZy1yYXRlcy5qc29uIgpVViA9IFsidXYiLCAicnVuIiwgIi0tbG9ja2VkIiwgIi0tZXh0cmEiLCAidHJhaW4iLCAiLS1uby1lZGl0YWJsZSIsICJweXRob24iXQoKCmRlZiBzZXR0aW5ncygpIC0+IHR1cGxlW2ludCwgaW50LCBzdHIsIHN0cl06CiAgICBudW1iZXJzID0gW29zLmVudmlyb25bbmFtZV0gZm9yIG5hbWUgaW4gKAogICAgICAgICJUT0tFTklaQVRJT05fQ09BUlNFX05VTV9TSEFSRFMiLCAiVE9LRU5JWkFUSU9OX0NPQVJTRV9TSEFSRF9JTkRFWCIpXQogICAgaWYgYW55KHJlLmZ1bGxtYXRjaChyIjB8WzEtOV1bMC05XSoiLCB2YWx1ZSkgaXMgTm9uZSBmb3IgdmFsdWUgaW4gbnVtYmVycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2hhcmQgY29udHJvbHMgbXVzdCBiZSBjYW5vbmljYWwgbm9ubmVnYXRpdmUgaW50ZWdlcnMiKQogICAgY291bnQsIGluZGV4ID0gbWFwKGludCwgbnVtYmVycykKICAgIGhhc2hlcyA9IFtvcy5lbnZpcm9uW25hbWVdIGZvciBuYW1lIGluICgKICAgICAgICAiVE9LRU5JWkFUSU9OX0NPQVJTRV9NQU5JRkVTVF9TSEEyNTYiLCAiVE9LRU5JWkFUSU9OX0NPQVJTRV9MUl9TSEEyNTYiKV0KICAgIGlmIG5vdCAxIDw9IGNvdW50IDw9IDE2MiBvciBub3QgMCA8PSBpbmRleCA8IGNvdW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlJlcXVpcmUgMSA8PSBudW1fc2hhcmRzIDw9IDE2MiBhbmQgMCA8PSBzaGFyZF9pbmRleCA8IG51bV9zaGFyZHMiKQogICAgaWYgYW55KHJlLmZ1bGxtYXRjaChyIlswLTlhLWZdezY0fSIsIHZhbHVlKSBpcyBOb25lIGZvciB2YWx1ZSBpbiBoYXNoZXMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkJvdGggY2FsbGVyLXByb3ZpZGVkIGZyb3plbiBTSEEtMjU2IHZhbHVlcyBhcmUgcmVxdWlyZWQiKQogICAgcmV0dXJuIGNvdW50LCBpbmRleCwgKmhhc2hlcwoKCmRlZiB2ZXJpZmllZF9ieXRlcyhwYXRoOiBQYXRoLCBleHBlY3RlZDogc3RyLCBsaW1pdDogaW50KSAtPiBieXRlczoKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAgICAgICBwYXlsb2FkID0gaGFuZGxlLnJlYWQobGltaXQgKyAxKQogICAgaWYgbGVuKHBheWxvYWQpID4gbGltaXQgb3IgaGFzaGxpYi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkgIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlNpemUvaGFzaCBtaXNtYXRjaDoge3BhdGh9IikKICAgIHJldHVybiBwYXlsb2FkCgoKZGVmIGNoZWNrZWRfc2V0dXAoY29tbWFuZDogbGlzdFtzdHJdLCBwcm9qZWN0OiBQYXRoKSAtPiBOb25lOgogICAgY29tcGxldGVkID0gc3VicHJvY2Vzcy5ydW4oY29tbWFuZCwgY3dkPXByb2plY3QsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRpbWVvdXQ9MTIwMCwgY2hlY2s9RmFsc2UpCiAgICBwcmludChjb21wbGV0ZWQuc3Rkb3V0Wy02MDAwOl0sIGZsdXNoPVRydWUpCiAgICBpZiBjb21wbGV0ZWQucmV0dXJuY29kZToKICAgICAgICBwcmludChjb21wbGV0ZWQuc3RkZXJyWy02MDAwOl0sIGZsdXNoPVRydWUpCiAgICAgICAgY29tcGxldGVkLmNoZWNrX3JldHVybmNvZGUoKQogICAgcHJpbnQoIkNvbXBsZXRlZDoiLCAiICIuam9pbihjb21tYW5kWzoxMF0pLCBmbHVzaD1UcnVlKQoKCmRlZiBtYWluKCkgLT4gZGljdDoKICAgIGdsb2JhbCB0b2tlbml6YXRpb25fY29hcnNlX3Byb2Nlc3MKICAgIGNvdW50LCBpbmRleCwgbWFuaWZlc3Rfc2hhLCByYXRlc19zaGEgPSBzZXR0aW5ncygpCiAgICBidW5kbGUgPSB2ZXJpZmllZF9ieXRlcyhDT05URU5UX1JPT1QgLyBCVU5ETEVfTkFNRSwgQlVORExFX1NIQTI1NiwgMTYgKiAxMDI0KioyKQogICAgbWFuaWZlc3RfYnl0ZXMgPSB2ZXJpZmllZF9ieXRlcyhDT05URU5UX1JPT1QgLyBNQU5JRkVTVF9OQU1FLCBtYW5pZmVzdF9zaGEsIDE2ICogMTAyNCoqMikKICAgIHJhdGVzX2J5dGVzID0gdmVyaWZpZWRfYnl0ZXMoQ09OVEVOVF9ST09UIC8gUkFURVNfTkFNRSwgcmF0ZXNfc2hhLCAxNiAqIDEwMjQqKjIpCiAgICBtYW5pZmVzdCwgcmF0ZXMgPSBqc29uLmxvYWRzKG1hbmlmZXN0X2J5dGVzKSwganNvbi5sb2FkcyhyYXRlc19ieXRlcykKICAgIGlmIChtYW5pZmVzdC5nZXQoInN0YWdlIikgIT0gImNvYXJzZSIgb3IgbWFuaWZlc3QuZ2V0KCJ0cmFjayIpICE9ICJ0aGVvcmVtX2ZhaXRoZnVsIgogICAgICAgICAgICBvciBtYW5pZmVzdC5nZXQoInBhaXJfam9icyIpICE9IDE2MiBvciBtYW5pZmVzdC5nZXQoImxlYXJuZWRfY29uZGl0aW9ucyIpICE9IDMyNAogICAgICAgICAgICBvciBsZW4obWFuaWZlc3QuZ2V0KCJqb2JzIiwgW10pKSAhPSAxNjIgb3IgcmF0ZXMuZ2V0KCJzdGFnZSIpICE9ICJscl9zZWxlY3Rpb24iCiAgICAgICAgICAgIG9yIG1hbmlmZXN0LmdldCgic291cmNlIiwge30pLmdldCgibGVhcm5pbmdfcmF0ZV9zZWxlY3Rpb25fc2hhMjU2IikgIT0gcmF0ZXNfc2hhKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVcGxvYWRzIGRvIG5vdCBpZGVudGlmeSB0aGUgZnJvemVuIFRyYWNrIEEgY29hcnNlMTYyL0xSIGNoYWluIikKICAgIHByb2plY3QgPSBDT05URU5UX1JPT1QgLyBmInRva2VuaXphdGlvbi1jb2Fyc2Utc2hhcmQte2luZGV4fS1vZi17Y291bnR9IgogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoaW8uQnl0ZXNJTyhidW5kbGUpKSBhcyBhcmNoaXZlOgogICAgICAgIGluZm9zID0gYXJjaGl2ZS5pbmZvbGlzdCgpCiAgICAgICAgaWYgbGVuKHtpbmZvLmZpbGVuYW1lIGZvciBpbmZvIGluIGluZm9zfSkgIT0gbGVuKGluZm9zKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIFpJUCBtZW1iZXJzIikKICAgICAgICBpZiBzdW0oaW5mby5maWxlX3NpemUgZm9yIGluZm8gaW4gaW5mb3MpID4gMTI4ICogMTAyNCoqMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRXhwYW5kZWQgcmVsZWFzZSBleGNlZWRzIHNpemUgbGltaXQiKQogICAgICAgIGZvciBpbmZvIGluIGluZm9zOgogICAgICAgICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChpbmZvLmZpbGVuYW1lKQogICAgICAgICAgICBub3JtYWxpemVkID0gc3RyKHBhdGgpICsgKCIvIiBpZiBpbmZvLmlzX2RpcigpIGVsc2UgIiIpCiAgICAgICAgICAgIGlmIChwYXRoLmlzX2Fic29sdXRlKCkgb3IgIi4uIiBpbiBwYXRoLnBhcnRzIG9yICJcXCIgaW4gaW5mby5maWxlbmFtZQogICAgICAgICAgICAgICAgICAgIG9yIG5vcm1hbGl6ZWQgIT0gaW5mby5maWxlbmFtZSBvciBpbmZvLmZsYWdfYml0cyAmIDEKICAgICAgICAgICAgICAgICAgICBvciBzdGF0LlNfSUZNVChpbmZvLmV4dGVybmFsX2F0dHIgPj4gMTYpIG5vdCBpbiAoMCwgc3RhdC5TX0lGUkVHLCBzdGF0LlNfSUZESVIpKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJVbnNhZmUgWklQIG1lbWJlcjoge2luZm8uZmlsZW5hbWV9IikKICAgICAgICBwcm9qZWN0Lm1rZGlyKGV4aXN0X29rPUZhbHNlKQogICAgICAgIGFyY2hpdmUuZXh0cmFjdGFsbChwcm9qZWN0KQogICAgZm9yIHJlbGF0aXZlLCBwYXlsb2FkIGluICgoTUFOSUZFU1RfUEFUSCwgbWFuaWZlc3RfYnl0ZXMpLCAoUkFURVNfUEFUSCwgcmF0ZXNfYnl0ZXMpKToKICAgICAgICBkZXN0aW5hdGlvbiA9IHByb2plY3QgLyByZWxhdGl2ZQogICAgICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgd2l0aCBkZXN0aW5hdGlvbi5vcGVuKCJ4YiIpIGFzIGhhbmRsZToKICAgICAgICAgICAgaGFuZGxlLndyaXRlKHBheWxvYWQpCiAgICBjaGVja2VkX3NldHVwKFtzeXMuZXhlY3V0YWJsZSwgIi1tIiwgInBpcCIsICJpbnN0YWxsIiwgIi1xIiwgInV2PT0wLjkuMjkiXSwgcHJvamVjdCkKICAgIGNoZWNrZWRfc2V0dXAoWyJ1diIsICJzeW5jIiwgIi0tbG9ja2VkIiwgIi0tZXh0cmEiLCAidGVzdCIsICItLW5vLWVkaXRhYmxlIiwKICAgICAgICAgICAgICAgICAgICItLXJlaW5zdGFsbC1wYWNrYWdlIiwgInRva2VuaXphdGlvbi1zdHVkeSJdLCBwcm9qZWN0KQogICAgIyBUaGUgZnJvemVuIHBhY2thZ2UgdmVyaWZpZXMgYm90aCBwYXJzZWQgYnVmZmVycyBhbmQgcmVjb25zdHJ1Y3RzIGFsbCAxNjIKICAgICMgam9icyBmcm9tIHRoZSBzZWxlY3RlZCByYXRlcy4gUGlsb3Qgb3V0Y29tZXMgYXJlIG5vdCBuZWVkZWQgb24gdGhpcyBob3N0LgogICAgZ2F0ZSA9ICIiImltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdG9rZW5pemF0aW9uX3N0dWR5LmV4ZWN1dGVfcmVnaXN0cnkgaW1wb3J0IGxvYWRfZnJvemVuX3JlZ2lzdHJ5CmZyb20gdG9rZW5pemF0aW9uX3N0dWR5LnJlZ2lzdHJ5IGltcG9ydCBidWlsZF9jb2Fyc2VfcmVnaXN0cnkKYWN0dWFsID0gbG9hZF9mcm96ZW5fcmVnaXN0cnkoUGF0aChzeXMuYXJndlsxXSksIGV4cGVjdGVkX3NoYTI1Nj1zeXMuYXJndlsyXSkKZXhwZWN0ZWQgPSBidWlsZF9jb2Fyc2VfcmVnaXN0cnkodHJhY2s9J3RoZW9yZW1fZmFpdGhmdWwnLAogICAgbGVhcm5pbmdfcmF0ZV9zZWxlY3Rpb25fcGF0aD1QYXRoKHN5cy5hcmd2WzNdKSwgZXhwZWN0ZWRfbGVhcm5pbmdfcmF0ZXNfc2hhMjU2PXN5cy5hcmd2WzRdLAogICAgZGF0YXNldF9yb290PVBhdGgoJ2FydGlmYWN0cy9kYXRhc2V0cycpLCBvdXRwdXRfcm9vdD1QYXRoKCdhcnRpZmFjdHMvcmVzdWx0cycpKQppZiBhY3R1YWwgIT0gZXhwZWN0ZWQ6CiAgICByYWlzZSBWYWx1ZUVycm9yKCdDb2Fyc2UgbWFuaWZlc3QgZG9lcyBub3QgZXF1YWwgY2Fub25pY2FsIHJlY29uc3RydWN0aW9uJykKcHJpbnQoJ1ZlcmlmaWVkIGNhbm9uaWNhbCBUcmFjayBBIGNvYXJzZSByZWdpc3RyeTonLCBsZW4oYWN0dWFsWydqb2JzJ10pLCAncGFpcnMnKQoiIiIKICAgIGNoZWNrZWRfc2V0dXAoWypVViwgIi1jIiwgZ2F0ZSwgTUFOSUZFU1RfUEFUSCwgbWFuaWZlc3Rfc2hhLCBSQVRFU19QQVRILCByYXRlc19zaGFdLCBwcm9qZWN0KQogICAgZm9yIHN1ZmZpeCBpbiAoWyJydWZmIiwgImNoZWNrIiwgInNyYyIsICJ0ZXN0cyJdLCBbInB5dGhvbiIsICItbSIsICJweXRlc3QiXSk6CiAgICAgICAgY2hlY2tlZF9zZXR1cChbInV2IiwgInJ1biIsICItLWxvY2tlZCIsICItLWV4dHJhIiwgInRlc3QiLCAiLS1uby1lZGl0YWJsZSIsICpzdWZmaXhdLCBwcm9qZWN0KQogICAgY2hlY2tlZF9zZXR1cChbKlVWLCAiLWMiLCAoImltcG9ydCB0b3JjaDsgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk7ICIKICAgICAgICAgICAgICAgICAgICJhc3NlcnQgJ1Q0JyBpbiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKTsgIgogICAgICAgICAgICAgICAgICAgInByaW50KHRvcmNoLl9fdmVyc2lvbl9fLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSkiKV0sIHByb2plY3QpCiAgICBwcm9ncmVzcyA9IGYiYXJ0aWZhY3RzL3JlZ2lzdHJ5L3Byb2dyZXNzL2NvYXJzZS1zaGFyZC17aW5kZXh9LW9mLXtjb3VudH0uanNvbiIKICAgIGNvbW1hbmQgPSBbKlVWLCAiLW0iLCAidG9rZW5pemF0aW9uX3N0dWR5LmV4ZWN1dGVfcmVnaXN0cnkiLCAiLS1tYW5pZmVzdCIsIE1BTklGRVNUX1BBVEgsCiAgICAgICAgICAgICAgICItLWV4cGVjdGVkLXNoYTI1NiIsIG1hbmlmZXN0X3NoYSwgIi0tbnVtLXNoYXJkcyIsIHN0cihjb3VudCksICItLXNoYXJkLWluZGV4Iiwgc3RyKGluZGV4KSwKICAgICAgICAgICAgICAgIi0tcHJlcGFyZS1kYXRhc2V0cyIsICItLXByb2dyZXNzIiwgcHJvZ3Jlc3NdCiAgICB3aXRoIChwcm9qZWN0IC8gImNvYXJzZS1leGVjdXRpb24ubG9nIikub3BlbigieCIpIGFzIGxvZzoKICAgICAgICB0b2tlbml6YXRpb25fY29hcnNlX3Byb2Nlc3MgPSBzdWJwcm9jZXNzLlBvcGVuKGNvbW1hbmQsIGN3ZD1wcm9qZWN0LCBzdGRvdXQ9bG9nLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpCiAgICBzdGF0ZSA9IHsic2hhcmQiOiBpbmRleCwgIm51bV9zaGFyZHMiOiBjb3VudCwgInBpZCI6IHRva2VuaXphdGlvbl9jb2Fyc2VfcHJvY2Vzcy5waWQsCiAgICAgICAgICAgICAicHJvamVjdCI6IHN0cihwcm9qZWN0KSwgInByb2dyZXNzIjogcHJvZ3Jlc3MsICJjb21tYW5kIjogY29tbWFuZCwKICAgICAgICAgICAgICJidW5kbGVfc2hhMjU2IjogQlVORExFX1NIQTI1NiwgIm1hbmlmZXN0X3NoYTI1NiI6IG1hbmlmZXN0X3NoYSwKICAgICAgICAgICAgICJsZWFybmluZ19yYXRlc19zaGEyNTYiOiByYXRlc19zaGF9CiAgICB3aXRoIChwcm9qZWN0IC8gImNvYXJzZS1wcm9jZXNzLmpzb24iKS5vcGVuKCJ4IikgYXMgaGFuZGxlOgogICAgICAgIGpzb24uZHVtcChzdGF0ZSwgaGFuZGxlLCBpbmRlbnQ9MikKICAgIHByaW50KGpzb24uZHVtcHMoc3RhdGUsIGluZGVudD0yKSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBzdGF0ZQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'}, 'colab-cli-coarse-collect.py': {'sha256': '7b14309d67819733a7c4171167c25a48a80927528e1a60d8874f3ec668adc456', 'base64': 'IiIiUGFja2FnZSBvbmUgY29tcGxldGVkIGNvYXJzZSBzaGFyZCBpbiB0aGUgbGF1bmNoZXIncyB1bmNoYW5nZWQgQ29sYWIga2VybmVsLiIiIgoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKCkNPTlRFTlRfUk9PVCA9IFBhdGgoIi9jb250ZW50IikKQlVORExFX1NIQTI1NiA9ICI4NWM0NGFmMDdmNmU3NTUzN2I5NmI5YjFiZWZhZTRiNGI1ODRlN2JhOGQ5YTI3MzJlMjkwZDg4ZWY4YmM3ZWZkIgpNQU5JRkVTVF9QQVRIID0gImFydGlmYWN0cy9yZWdpc3RyeS90aGVvcmVtLWZhaXRoZnVsLWNvYXJzZS5qc29uIgpSQVRFU19QQVRIID0gImFydGlmYWN0cy9yZWdpc3RyeS90aGVvcmVtLWZhaXRoZnVsLWxlYXJuaW5nLXJhdGVzLmpzb24iCgoKZGVmIHZlcmlmaWVkX2pzb24ocGF0aDogUGF0aCwgZXhwZWN0ZWQ6IHN0cikgLT4gZGljdDoKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIGhhbmRsZToKICAgICAgICBwYXlsb2FkID0gaGFuZGxlLnJlYWQoMTYgKiAxMDI0KioyICsgMSkKICAgIGlmIGxlbihwYXlsb2FkKSA+IDE2ICogMTAyNCoqMiBvciBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3QoKSAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU2l6ZS9oYXNoIG1pc21hdGNoOiB7cGF0aH0iKQogICAgcmV0dXJuIGpzb24ubG9hZHMocGF5bG9hZCkKCgpkZWYgbWFpbigpIC0+IGRpY3Q6CiAgICB2YWx1ZXMgPSBbb3MuZW52aXJvbltrZXldIGZvciBrZXkgaW4gKAogICAgICAgICJUT0tFTklaQVRJT05fQ09BUlNFX05VTV9TSEFSRFMiLCAiVE9LRU5JWkFUSU9OX0NPQVJTRV9TSEFSRF9JTkRFWCIpXQogICAgaWYgYW55KHJlLmZ1bGxtYXRjaChyIjB8WzEtOV1bMC05XSoiLCB2YWx1ZSkgaXMgTm9uZSBmb3IgdmFsdWUgaW4gdmFsdWVzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTaGFyZCBjb250cm9scyBtdXN0IGJlIGNhbm9uaWNhbCBub25uZWdhdGl2ZSBpbnRlZ2VycyIpCiAgICBjb3VudCwgaW5kZXggPSBtYXAoaW50LCB2YWx1ZXMpCiAgICBtYW5pZmVzdF9zaGEgPSBvcy5lbnZpcm9uWyJUT0tFTklaQVRJT05fQ09BUlNFX01BTklGRVNUX1NIQTI1NiJdCiAgICByYXRlc19zaGEgPSBvcy5lbnZpcm9uWyJUT0tFTklaQVRJT05fQ09BUlNFX0xSX1NIQTI1NiJdCiAgICBpZiAobm90IDEgPD0gY291bnQgPD0gMTYyIG9yIG5vdCAwIDw9IGluZGV4IDwgY291bnQKICAgICAgICAgICAgb3IgYW55KHJlLmZ1bGxtYXRjaChyIlswLTlhLWZdezY0fSIsIHZhbHVlKSBpcyBOb25lIGZvciB2YWx1ZSBpbiAobWFuaWZlc3Rfc2hhLCByYXRlc19zaGEpKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCB0cmFuc3BvcnQgY29udHJvbHMgb3IgZnJvemVuIGFydGlmYWN0IGhhc2hlcyIpCiAgICBwcm9qZWN0ID0gQ09OVEVOVF9ST09UIC8gZiJ0b2tlbml6YXRpb24tY29hcnNlLXNoYXJkLXtpbmRleH0tb2Yte2NvdW50fSIKICAgIHByb2dyZXNzID0gZiJhcnRpZmFjdHMvcmVnaXN0cnkvcHJvZ3Jlc3MvY29hcnNlLXNoYXJkLXtpbmRleH0tb2Yte2NvdW50fS5qc29uIgogICAgY29tbWFuZCA9IFsidXYiLCAicnVuIiwgIi0tbG9ja2VkIiwgIi0tZXh0cmEiLCAidHJhaW4iLCAiLS1uby1lZGl0YWJsZSIsICJweXRob24iLCAiLW0iLAogICAgICAgICAgICAgICAidG9rZW5pemF0aW9uX3N0dWR5LmV4ZWN1dGVfcmVnaXN0cnkiLCAiLS1tYW5pZmVzdCIsIE1BTklGRVNUX1BBVEgsCiAgICAgICAgICAgICAgICItLWV4cGVjdGVkLXNoYTI1NiIsIG1hbmlmZXN0X3NoYSwgIi0tbnVtLXNoYXJkcyIsIHN0cihjb3VudCksICItLXNoYXJkLWluZGV4Iiwgc3RyKGluZGV4KSwKICAgICAgICAgICAgICAgIi0tcHJlcGFyZS1kYXRhc2V0cyIsICItLXByb2dyZXNzIiwgcHJvZ3Jlc3NdCiAgICBwcm9jZXNzID0gdG9rZW5pemF0aW9uX2NvYXJzZV9wcm9jZXNzICAjIG5vcWE6IEY4MjEKICAgIGV4cGVjdGVkX3N0YXRlID0geyJzaGFyZCI6IGluZGV4LCAibnVtX3NoYXJkcyI6IGNvdW50LCAicGlkIjogcHJvY2Vzcy5waWQsCiAgICAgICAgICAgICAgICAgICAgICAicHJvamVjdCI6IHN0cihwcm9qZWN0KSwgInByb2dyZXNzIjogcHJvZ3Jlc3MsICJjb21tYW5kIjogY29tbWFuZCwKICAgICAgICAgICAgICAgICAgICAgICJidW5kbGVfc2hhMjU2IjogQlVORExFX1NIQTI1NiwgIm1hbmlmZXN0X3NoYTI1NiI6IG1hbmlmZXN0X3NoYSwKICAgICAgICAgICAgICAgICAgICAgICJsZWFybmluZ19yYXRlc19zaGEyNTYiOiByYXRlc19zaGF9CiAgICBzdGF0ZSA9IGpzb24ubG9hZHMoKHByb2plY3QgLyAiY29hcnNlLXByb2Nlc3MuanNvbiIpLnJlYWRfYnl0ZXMoKSkKICAgIHJldHVybmNvZGUgPSBwcm9jZXNzLnBvbGwoKQogICAgaWYgdHlwZShyZXR1cm5jb2RlKSBpcyBub3QgaW50IG9yIHJldHVybmNvZGUgIT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNvYXJzZSBwcm9jZXNzIGlzIHJ1bm5pbmcgb3IgZmFpbGVkOyBwcmVzZXJ2ZSBydW50aW1lIGFuZCBsb2ciKQogICAgaWYgc3RhdGUgIT0gZXhwZWN0ZWRfc3RhdGUgb3IgcHJvY2Vzcy5hcmdzICE9IGNvbW1hbmQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTGl2ZSBwcm9jZXNzL2NvbW1hbmQgZG9lcyBub3QgbWF0Y2ggdGhlIGZyb3plbiBzaGFyZCBpZGVudGl0eSIpCiAgICBtYW5pZmVzdCA9IHZlcmlmaWVkX2pzb24ocHJvamVjdCAvIE1BTklGRVNUX1BBVEgsIG1hbmlmZXN0X3NoYSkKICAgIHJhdGVzID0gdmVyaWZpZWRfanNvbihwcm9qZWN0IC8gUkFURVNfUEFUSCwgcmF0ZXNfc2hhKQogICAgaWYgKG1hbmlmZXN0LmdldCgic3RhZ2UiKSAhPSAiY29hcnNlIiBvciBtYW5pZmVzdC5nZXQoInRyYWNrIikgIT0gInRoZW9yZW1fZmFpdGhmdWwiCiAgICAgICAgICAgIG9yIG1hbmlmZXN0LmdldCgicGFpcl9qb2JzIikgIT0gMTYyIG9yIGxlbihtYW5pZmVzdC5nZXQoImpvYnMiLCBbXSkpICE9IDE2MgogICAgICAgICAgICBvciBtYW5pZmVzdC5nZXQoImxlYXJuZWRfY29uZGl0aW9ucyIpICE9IDMyNAogICAgICAgICAgICBvciBtYW5pZmVzdC5nZXQoInNvdXJjZSIsIHt9KS5nZXQoImxlYXJuaW5nX3JhdGVfc2VsZWN0aW9uX3NoYTI1NiIpICE9IHJhdGVzX3NoYQogICAgICAgICAgICBvciByYXRlcy5nZXQoInN0YWdlIikgIT0gImxyX3NlbGVjdGlvbiIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZyb3plbiBjb2Fyc2UvTFIgcHJvdmVuYW5jZSBpcyBpbmNvbnNpc3RlbnQiKQogICAgc2VsZWN0ZWQgPSBtYW5pZmVzdFsiam9icyJdW2luZGV4Ojpjb3VudF0KICAgIGlkcyA9IFtqb2JbImpvYl9pZCJdIGZvciBqb2IgaW4gc2VsZWN0ZWRdCiAgICBsZWRnZXIgPSBqc29uLmxvYWRzKChwcm9qZWN0IC8gcHJvZ3Jlc3MpLnJlYWRfYnl0ZXMoKSkKICAgIGV4cGVjdGVkX2xlZGdlciA9IHsic2NoZW1hX3ZlcnNpb24iOiAxLCAibWFuaWZlc3Rfc2hhMjU2IjogbWFuaWZlc3Rfc2hhLCAic2hhcmRfaW5kZXgiOiBpbmRleCwKICAgICAgICAgICAgICAgICAgICAgICAibnVtX3NoYXJkcyI6IGNvdW50LCAiZHJ5X3J1biI6IEZhbHNlLCAiZmFpbGVkX2pvYiI6IE5vbmUsICJzZWxlY3RlZF9qb2JzIjogaWRzLAogICAgICAgICAgICAgICAgICAgICAgICJza2lwcGVkX2pvYnMiOiBbXSwgInBsYW5uZWRfam9icyI6IFtdLAogICAgICAgICAgICAgICAgICAgICAgICJwcmVwYXJlZF9kYXRhc2V0cyI6IFtkYXRhc2V0WyJwYXRoIl0gZm9yIGRhdGFzZXQgaW4gbWFuaWZlc3RbImRhdGFzZXRzIl1dLAogICAgICAgICAgICAgICAgICAgICAgICJjb21wbGV0ZWRfam9icyI6IGxlZGdlci5nZXQoImNvbXBsZXRlZF9qb2JzIil9CiAgICBkb25lID0gbGVkZ2VyLmdldCgiY29tcGxldGVkX2pvYnMiKQogICAgaWYgbGVkZ2VyICE9IGV4cGVjdGVkX2xlZGdlciBvciBub3QgaXNpbnN0YW5jZShkb25lLCBsaXN0KSBvciBsZW4oZG9uZSkgIT0gbGVuKGlkcyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTGVkZ2VyIGRvZXMgbm90IHByb3ZlIHRoZSBjb21wbGV0ZSBmcmVzaCBzaGFyZCIpCiAgICBmb3Igb3JkaW5hbCwgKGVudHJ5LCBqb2JfaWQpIGluIGVudW1lcmF0ZSh6aXAoZG9uZSwgaWRzLCBzdHJpY3Q9VHJ1ZSksIDEpOgogICAgICAgIGlmIChzZXQoZW50cnkpICE9IHsiam9iX2lkIiwgIm9yZGluYWwiLCAiZWxhcHNlZF9zZWNvbmRzIn0gb3IgZW50cnlbImpvYl9pZCJdICE9IGpvYl9pZAogICAgICAgICAgICAgICAgb3IgdHlwZShlbnRyeVsib3JkaW5hbCJdKSBpcyBub3QgaW50IG9yIGVudHJ5WyJvcmRpbmFsIl0gIT0gb3JkaW5hbAogICAgICAgICAgICAgICAgb3IgdHlwZShlbnRyeVsiZWxhcHNlZF9zZWNvbmRzIl0pIG5vdCBpbiAoaW50LCBmbG9hdCkKICAgICAgICAgICAgICAgIG9yIG5vdCBtYXRoLmlzZmluaXRlKGVudHJ5WyJlbGFwc2VkX3NlY29uZHMiXSkgb3IgZW50cnlbImVsYXBzZWRfc2Vjb25kcyJdIDw9IDApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNYWxmb3JtZWQgY29tcGxldGVkLWpvYiBsZWRnZXIgZW50cnkiKQogICAgbWVtYmVycyA9IFtQYXRoKHByb2dyZXNzKSwgUGF0aCgiY29hcnNlLWV4ZWN1dGlvbi5sb2ciKSwgUGF0aCgiY29hcnNlLXByb2Nlc3MuanNvbiIpXQogICAgZm9yIGpvYiBpbiBzZWxlY3RlZDoKICAgICAgICBvdXRwdXQgPSBQYXRoKGpvYlsib3V0cHV0Il0pCiAgICAgICAgbWVtYmVycy5leHRlbmQoW291dHB1dCwgb3V0cHV0LndpdGhfc3VmZml4KCIubGVmdC5wdCIpLCBvdXRwdXQud2l0aF9zdWZmaXgoIi5yaWdodC5wdCIpXSkKICAgIGlmIGxlbihzZXQobWVtYmVycykpICE9IGxlbihtZW1iZXJzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUgYXJjaGl2ZSBtZW1iZXJzIikKICAgIGZvciByZWxhdGl2ZSBpbiBtZW1iZXJzOgogICAgICAgIHBhdGggPSBwcm9qZWN0IC8gcmVsYXRpdmUKICAgICAgICBpZiAocmVsYXRpdmUuaXNfYWJzb2x1dGUoKSBvciAiLi4iIGluIHJlbGF0aXZlLnBhcnRzIG9yICJcXCIgaW4gc3RyKHJlbGF0aXZlKQogICAgICAgICAgICAgICAgb3Igc3RyKFB1cmVQb3NpeFBhdGgocmVsYXRpdmUpKSAhPSBzdHIocmVsYXRpdmUpIG9yIG5vdCBwYXRoLmlzX2ZpbGUoKQogICAgICAgICAgICAgICAgb3IgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGgucmVzb2x2ZSgpLmlzX3JlbGF0aXZlX3RvKHByb2plY3QucmVzb2x2ZSgpKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNaXNzaW5nIG9yIHVuc2FmZSByZXN1bHQgZmlsZToge3JlbGF0aXZlfSIpCiAgICBoYXJkd2FyZSA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsibnZpZGlhLXNtaSIsICItLXF1ZXJ5LWdwdT1uYW1lLG1lbW9yeS50b3RhbCxkcml2ZXJfdmVyc2lvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyIl0sIHRleHQ9VHJ1ZSkuc3RyaXAoKQogICAgaWYgIlQ0IiBub3QgaW4gaGFyZHdhcmU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29sbGVjdG9yIGlzIG5vdCBydW5uaW5nIG9uIHRoZSByZWdpc3RlcmVkIFQ0IikKICAgIGFyY2hpdmVfcGF0aCA9IENPTlRFTlRfUk9PVCAvIGYidG9rZW5pemF0aW9uLWNvYXJzZS1yZXR1cm4te2luZGV4fS1vZi17Y291bnR9LnppcCIKICAgIHdpdGggYXJjaGl2ZV9wYXRoLm9wZW4oInhiKyIpIGFzIGhhbmRsZToKICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShoYW5kbGUsICJ3IiwgY29tcHJlc3Npb249emlwZmlsZS5aSVBfREVGTEFURUQpIGFzIGFyY2hpdmU6CiAgICAgICAgICAgIGZvciByZWxhdGl2ZSBpbiBtZW1iZXJzOgogICAgICAgICAgICAgICAgYXJjaGl2ZS53cml0ZShwcm9qZWN0IC8gcmVsYXRpdmUsIGFyY25hbWU9cmVsYXRpdmUuYXNfcG9zaXgoKSkKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZXN0cigiY29hcnNlLWhhcmR3YXJlLmpzb24iLCBqc29uLmR1bXBzKHsibnZpZGlhX3NtaSI6IGhhcmR3YXJlfSwgaW5kZW50PTIpKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHNpemUgPSBoYW5kbGUudGVsbCgpCiAgICAgICAgaGFuZGxlLnNlZWsoMCkKICAgICAgICBkaWdlc3QgPSBoYXNobGliLmZpbGVfZGlnZXN0KGhhbmRsZSwgInNoYTI1NiIpLmhleGRpZ2VzdCgpCiAgICBzdW1tYXJ5ID0geyJzaGFyZCI6IGluZGV4LCAibnVtX3NoYXJkcyI6IGNvdW50LCAiY29tcGxldGVkX3BhaXJzIjogbGVuKHNlbGVjdGVkKSwKICAgICAgICAgICAgICAgImNoZWNrcG9pbnRfZmlsZXMiOiAyICogbGVuKHNlbGVjdGVkKSwgImFyY2hpdmUiOiBzdHIoYXJjaGl2ZV9wYXRoKSwKICAgICAgICAgICAgICAgImFyY2hpdmVfYnl0ZXMiOiBzaXplLCAiYXJjaGl2ZV9zaGEyNTYiOiBkaWdlc3QsICJoYXJkd2FyZSI6IGhhcmR3YXJlLAogICAgICAgICAgICAgICAibWFuaWZlc3Rfc2hhMjU2IjogbWFuaWZlc3Rfc2hhLCAibGVhcm5pbmdfcmF0ZXNfc2hhMjU2IjogcmF0ZXNfc2hhfQogICAgcHJpbnQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MiksIGZsdXNoPVRydWUpCiAgICByZXR1cm4gc3VtbWFyeQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'}}


In [ ]:
import os
import subprocess
from datetime import datetime, timezone
from uuid import uuid4
hardware = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True).strip()
validate_hardware(hardware)
print('Hardware:', hardware)
CONTENT_ROOT = Path('/content')
project_path = CONTENT_ROOT / f'tokenization-coarse-shard-{SHARD_INDEX}-of-{NUM_SHARDS}'
if project_path.exists():
    raise FileExistsError(f'Preserving existing run at {project_path}. Do not rerun setup or launch. If the same kernel is alive, rerun only the monitor/collect cells.')
install_assets(EMBEDDED_ASSETS, CONTENT_ROOT)
os.environ.update({'TOKENIZATION_COARSE_NUM_SHARDS': str(NUM_SHARDS), 'TOKENIZATION_COARSE_SHARD_INDEX': str(SHARD_INDEX), 'TOKENIZATION_COARSE_MANIFEST_SHA256': EMBEDDED_ASSETS['tokenization-coarse-manifest.json']['sha256'], 'TOKENIZATION_COARSE_LR_SHA256': EMBEDDED_ASSETS['tokenization-coarse-learning-rates.json']['sha256'], 'UV_PYTHON': '3.12'})
print('All five frozen inputs verified. Training has not started yet.')


In [ ]:
drive_results = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:12]
    drive_results = Path('/content/drive/MyDrive/tokenization-results') / f'coarse-shard-{SHARD_INDEX}-of-{NUM_SHARDS}-{run_id}'
    drive_results.mkdir(parents=True, exist_ok=False)
    print('Drive backup folder:', drive_results)
else:
    print('Drive backup disabled. Download results before the runtime disappears.')


## Install, check, and launch once

This cell runs the **unchanged frozen launcher** in this kernel and retains its `Popen` handle.
Do not use `!python` to launch it. Do not rerun this cell to resume a run.
The frozen package checks must pass and the GPU must be T4 before training starts.

In [ ]:
launcher_namespace = load_adapter(CONTENT_ROOT / 'colab-cli-coarse.py', EMBEDDED_ASSETS['colab-cli-coarse.py']['sha256'])
print('Installing locked environment and checking frozen inputs; this may take several minutes.', flush=True)
launch_state = launcher_namespace['main']()
tokenization_coarse_process = launcher_namespace['tokenization_coarse_process']
last_backed_up = 0
print('Training launched. Log:', Path(launch_state['project']) / 'coarse-execution.log')


## Progress and optional Drive snapshots

Progress is printed every 30 seconds. A newly observed completed-pair count triggers a Drive snapshot.
These snapshots do not include unfinished pairs and are **not validated final archives**.
If this cell is interrupted, the child process may still be running; keep the same kernel
and rerun **only this monitor cell**, not setup or launch. A failed Drive backup is reported
but does not stop training or block the final local download. Existing incomplete copies are preserved.
This notebook does not bypass Colab runtime limits or recover a lost kernel automatically.

In [ ]:
import time
project = Path(launch_state['project'])
progress_path = project / launch_state['progress']
last_backup_attempt = last_backed_up
while True:
    returncode = tokenization_coarse_process.poll()
    if progress_path.is_file():
        progress_bytes = progress_path.read_bytes()
        progress = json.loads(progress_bytes)
        completed_count = len(progress.get('completed_jobs', []))
        print(f'Completed paired jobs: {completed_count}/9; process status: {returncode}', flush=True)
        if drive_results is not None and completed_count > last_backup_attempt:
            last_backup_attempt = completed_count
            try:
                backup_path = backup_completed(project, launch_state, drive_results, ledger_bytes=progress_bytes)
                last_backed_up = completed_count
                print('Saved partial/unvalidated backup:', backup_path, flush=True)
            except (OSError, ValueError) as error:
                print('BACKUP FAILED:', error, flush=True)
                print('Training continues. This snapshot is NOT safely saved; preserve the runtime and download final results.', flush=True)
    else:
        print('Preparing datasets / waiting for first progress ledger...', flush=True)
    if returncode is not None:
        if returncode != 0:
            log = project / 'coarse-execution.log'
            print(log.read_text(errors='replace')[-12000:] if log.is_file() else 'No log available')
            raise RuntimeError(f'Training failed with exit code {returncode}; preserve the runtime and log.')
        break
    time.sleep(30)
print('Process exited successfully. Next: verify completion and package results.')


## Collect and persist results

The unchanged collector requires all nine fresh jobs, all eighteen checkpoints, the exact ledger,
the original process handle, and T4 hardware. Collection does **not** replace independent scientific
validation after import. Drive copies are attempted and hash-checked before the browser download.
If a Drive copy fails, the error is printed and the verified local files remain downloadable.
If the browser download fails, rerun **only the last download cell**.

In [ ]:
collector_namespace = load_adapter(CONTENT_ROOT / 'colab-cli-coarse-collect.py', EMBEDDED_ASSETS['colab-cli-coarse-collect.py']['sha256'], {'tokenization_coarse_process': tokenization_coarse_process})
result_summary = collector_namespace['main']()
summary_path = CONTENT_ROOT / f'tokenization-coarse-return-{SHARD_INDEX}-of-{NUM_SHARDS}-summary.json'
write_unchanged(summary_path, (json.dumps(result_summary, indent=2) + '\n').encode())
print('Final archive and summary created. Continue to persistent save, then download.')


In [ ]:
if drive_results is not None:
    try:
        save_final(result_summary, drive_results)
        print('Final archive and summary saved and verified in Drive:', drive_results)
    except (OSError, ValueError) as error:
        print('BACKUP FAILED:', error)
        print('Drive persistence is NOT confirmed. Download the local archive and summary in the next cell now.')
else:
    print('No Drive copy requested. Download both files in the next cell now.')


In [ ]:
from google.colab import files
if file_sha256(Path(result_summary['archive'])) != result_summary['archive_sha256']:
    raise ValueError('Final archive hash mismatch; preserve the runtime for inspection')
files.download(result_summary['archive'])
files.download(str(summary_path))
print('Keep the ZIP and summary together, and return them for independent validation.')


## If something stops

- No T4 or an older driver: change the runtime type/version if available; keep the frozen dependency pins unchanged.
- Setup/test failure: preserve the full error output. Do not delete the existing project directory or bypass checks.
- A monitor interruption: rerun only the monitor in the same kernel. Resetting the kernel loses the required process handle.
- A lost runtime: recover whatever partial ZIPs reached Drive. They need separate inspection; this notebook cannot automatically resume them.
- A failed download: use the last cell again, or obtain the final ZIP and summary from the printed Drive folder.
- To run another shard: save this run first, then use a fresh runtime and deliberately choose another missing index from 3–17.

Reference: [Colab FAQ](https://research.google.com/colaboratory/faq.html).
CUDA driver policy: [NVIDIA minor-version compatibility](https://docs.nvidia.com/deploy/cuda-compatibility/minor-version-compatibility.html).